# 리포트 07 — 완전파는 정확도의 과녁이고, SBR+PO 는 표를 만들 수 있는 비용대에서 가장 정확하다

> ### 한 일
> **σ 를 조달하는 다섯 갈래를 비용과 정확도로 지도에 놓고, 게재된 반론 하나를 그대로 실어 우리 선택의 값을 적었다.**

### 결과
1. 갈래는 5개다. 완전파는 정확도의 과녁이고 — 우리 2D EFIE MoM 자체검사가 정확 원기둥 고유함수해 대비 0.00027 dB [^1] 다 — 그 대신 비용이 표를 못 만들게 한다.
2. 우리 커널은 자세 하나(방위·고도 한 점 × 반송파 하나 → σ 한 값)에 중앙값 38.1 ms [^2] 다(측정은 Γ(θ) 배선 전 batch 커널 경로). 같은 카드·같은 씬에서 스톡 `sionna.rt.PathSolver` 전파 해가 106.3 ms [^3] 이므로 하드웨어 변수가 제거된다.
3. ⚠ 반론도 그대로 싣는다 — Ziganshin arXiv:2604.05991v2 pp.1–2, 게재된 반론: 'the need to cascade PO after RT negates the computational advantages of RT' [^4]
4. 우리 구현에서 PO 적분은 광선캐스팅의 16.2 [^5]배다. 게재된 유일한 GPU 커널 분해는 같은 캐스케이드를 광선발사의 6.5% [^6] 로 적는다 — 절반은 우리 몫이다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 갈래 지도 | 방법마다 «무엇을 푸는가» 를 그 방법의 게재 문헌 표현으로 적는다 |
| 비용 인용 | 완전파 쪽 비용은 게재본 문장을 축자로 싣는다 — 우리 추정이 아니다 |
| 런타임 대조 | 같은 `RTX 4090`, 같은 챔버 씬에서 우리 커널과 스톡 솔버를 나란히 잰다 (⚠ 재는 양은 서로 다르다 — 전파 경로 대 σ) |

### 재현

```bash
PYTHONPATH=src python benchmark/build_report00_po_case.py
PYTHONPATH=src python src/build_part01_stock_engine.py
```

| | |
|---|---|
| 출력 | `outputs/report00_po_case.json` |
| 소요 | 약 1분 (GPU 0장 — JSON 읽기다) |

---

## 다섯 갈래의 지도

| 방법 | 무엇을 푸는가 |
|---|---|
| ① 완전파 (MoM / MLFMM / FDTD) | 맥스웰 방정식을 근사 없이 푼다. 크리핑파·다중산란·편파를 전부 포함한다. |
| ② SBR + PO (우리) | 광선으로 '어느 면이 실제로 조명되는가' 를 찾고, 그 면에서 PO 표면적분으로 σ 를 낸다. 상용 EM 솔버(FEKO/CST/HFSS SBR+)의 고주파 표준 방법이다. |
| ③ 통계 RCS 주입 (3GPP 표 조회) | σ 를 규격 표에서 읽어 각도 섹터로 조회하고 경로전력에 곱한다. |
| ④ 기하 대리표적 (큐브·박스·구) | 표적을 정육면체·직육면체·구로 바꾼다. 선행에서 가장 흔한 회피다. |
| ⑤ 실측 | 무향실·CATR 에서 직접 잰다. 절대 앵커의 최종 출처다. |

출처 [^7]

⭐ 우리 자리를 정확히 적는다. 그래픽 레이트레이서 위에 자기 PO 적분기를 얹는 것은 우리 발명이 아니라 **이 문제의 표준 대응**이고, 두 팀이 독립적으로 같은 곳에 도달했다. [^8]

## 완전파는 정확도의 과녁이다

우리 2D EFIE MoM 자체검사는 정확 원기둥 고유함수해 대비 0.00027 dB [^1] 다. 그 눈금이 «완전파가 참값이다» 를 이 저장소 안에서 실제로 붙잡아 준다.

그 대신 비용이 표를 못 만들게 한다. 게재본 문장 그대로 — it should be emphasized that MLFMM simulations are considerably more computationally demanding. For instance, each of the MLFMM simulations in this study took several hours. [^9]

우리 커널은 자세 하나에 중앙값 38.1 ms [^2] 다 — 측정은 Γ(θ) 배선 전 batch 커널 경로의 값이다. 같은 `RTX 4090`, 같은 챔버 씬에서 스톡 `sionna.rt.PathSolver` 전파 해가 106.3 ms [^3] 이므로 하드웨어 변수는 여기서 제거된다(⚠ 재는 양은 서로 다르다 — 전파 경로 대 σ).

## 게재된 반론 하나 — 캐스케이드 비용

⚠ 반론을 그대로 싣는다 — Ziganshin arXiv:2604.05991v2 pp.1–2, 게재된 반론: 'the need to cascade PO after RT negates the computational advantages of RT' [^4]

우리 구현에서 PO 적분은 광선캐스팅의 16.2 [^5]배다 — 적분이 아직 호스트 numpy 라서이고, 측정은 Γ(θ) 배선 전 batch 커널 경로다. 게재된 유일한 GPU 커널 분해(SagittaSBR)는 같은 캐스케이드를 광선발사의 6.5% [^6] 로 적는다. 절반은 우리 몫이다.

⭐ 그래서 이 편의 결론은 «PO 가 가장 정확하다» 가 아니라 «표를 만들 수 있는 비용대에서 가장 정확하다» 이다. 그 표가 실제로 얼마나 맞는지는 [편 21 «해석 PO 구 대비 구현오차는 kr 전 구간에서 0.201 dB 안이다»](21_kernel-vs-reference.ipynb) 가 잰다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| PO 적분을 디바이스 커널로 내린다 | 캐스케이드 반론의 비율 16.2 [^5]배가 게재된 GPU 분해 수준으로 내려가는지가 확정된다 | `src/rcs_sbr.py` |
| Γ(θ) 를 batch 경로에 배선한 뒤 runtime_benchmark 를 같은 카드에서 다시 돌린다 | 각도 모양이 붙은 커널의 자세당 비용이 확정된다 | `benchmark/runtime_benchmark.py` |
| 이 비용대에서 얻은 σ 를 해석 기준해와 맞댄다 | 구현오차가 kr 전 구간에서 dB 로 확정된다 | [편 21 «해석 PO 구 대비 구현오차는 kr 전 구간에…»](21_kernel-vs-reference.ipynb) |
| 남들이 이 다섯 갈래 중 무엇을 골랐는지 게재본에서 센다 | 조달처 일곱 갈래와 그 갈래가 사 주는 주장의 크기가 확정된다 | [편 10 «표적 서명을 어디서 조달했는지가 그 논문이 낼…»](10_procurement.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 9개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report00_po_case.json` | `s3_validation.layer3_thin_plate_2d_mom.mom_selftest_worst_db` | 0.0002705 |
| [^2] | `outputs/report00_po_case.json` | `s1_alternatives.ours_runtime.ours_per_pose_ms_median` | 38.13 |
| [^3] | `outputs/report00_po_case.json` | `s1_alternatives.stock_sionna_same_card.stock_sionna_ms_median` | 106.3 |
| [^4] | `outputs/report00_po_case.json` | `s1_alternatives.cascade_cost_objection._the_objection` | Ziganshin arXiv:2604.05991v2 pp.1–2, 게재된 반론: 'the need… |
| [^5] | `outputs/report00_po_case.json` | `s1_alternatives.cascade_cost_objection.our_po_over_rt` | 16.23 |
| [^6] | `outputs/report00_po_case.json` | `s1_alternatives.cascade_cost_objection.sagitta_po_over_raylaunch_A100_fp32` | 0.06498 |
| [^7] | `outputs/report00_po_case.json` | `s1_alternatives.alternatives` | (5행 표) |
| [^8] | `outputs/report00_po_case.json` | `s2_our_kernel.same_methodology_as.statement` | 그래픽 레이트레이서 위에 자기 PO 적분기를 얹는 것은 우리 발명이 아니라 **이 문제의 표준 대응… |
| [^9] | `outputs/report00_po_case.json` | `s1_alternatives.alternatives[0].cost_quote_mlfmm` | it should be emphasized that MLFMM simulations are cons… |